# N07 — PyTorch MLP Training Loop

**Sessions:** 45–47  
**Objective:** Create tensors/loaders/model, train and validate safely, and save the best checkpoint.

## Task contract
Run top-to-bottom from a fresh kernel. Do not install packages inside the notebook. Record one error or misconception and complete the independent transfer before submission.


In [ ]:
from __future__ import annotations
import hashlib, json, platform, random
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", platform.python_version())
print("Working directory:", Path.cwd())


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:',torch.__version__,'device:',device)
X,y=make_moons(n_samples=500,noise=.2,random_state=SEED)
Xtr,Xva,ytr,yva=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
train_loader=DataLoader(TensorDataset(torch.tensor(Xtr,dtype=torch.float32),torch.tensor(ytr,dtype=torch.long)),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
val_loader=DataLoader(TensorDataset(torch.tensor(Xva,dtype=torch.float32),torch.tensor(yva,dtype=torch.long)),batch_size=64)
model=nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,2)).to(device)
loss_fn=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(),lr=.02)


In [ ]:
def run_epoch(model,loader,training):
    model.train(training); total_loss=0.; correct=0; count=0
    context=torch.enable_grad() if training else torch.no_grad()
    with context:
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            if training: opt.zero_grad(set_to_none=True)
            logits=model(xb); loss=loss_fn(logits,yb)
            if training:
                loss.backward(); opt.step()
            total_loss += loss.item()*len(xb); correct += (logits.argmax(1)==yb).sum().item(); count += len(xb)
    return total_loss/count, correct/count

best=-1; checkpoint=Path('/tmp/noai_mlp_best.pt')
for epoch in range(1,7):
    tr=run_epoch(model,train_loader,True); va=run_epoch(model,val_loader,False)
    print(epoch,{'train_loss':round(tr[0],4),'train_acc':round(tr[1],3),'val_loss':round(va[0],4),'val_acc':round(va[1],3)})
    if va[1]>best:
        best=va[1]; torch.save({'model':model.state_dict(),'seed':SEED,'val_acc':best},checkpoint)
assert checkpoint.exists() and best>.75


## Worksheet
1. Why does CrossEntropyLoss expect class indices rather than one-hot labels here?
2. Explain `train()` versus `eval()`.
3. Why is `no_grad()` used in validation?
4. Trace tensor shapes through each layer.
5. Identify where device mismatch would occur.


## Independent transfer
Close the model cell and rebuild the complete train/validate/checkpoint workflow. Add early stopping or weight decay as one controlled experiment and explain the result.


## Fresh-run checklist

- [ ] Restart kernel and run all cells in order.
- [ ] Confirm assertions pass.
- [ ] Record package versions and seed.
- [ ] Save the required artifact with a relative path.
- [ ] Add an error-log entry and AI-use note.
- [ ] Explain one teacher-selected cell without reading it.
